<a href="https://colab.research.google.com/github/rdelhibabu/QFL_Edge/blob/main/QFL_Edge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this cell to install all dependencies and the Qiskit-PennyLane plugin
!pip install pennylane qiskit qiskit-aer pennylane-qiskit torch numpy matplotlib

import pennylane as qml
import torch
import numpy as np
import matplotlib.pyplot as plt

# Qiskit Aer for hardware noise modeling
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error, phase_damping_error

In [2]:
# Hardware and Network Parameters (Table 5)
NUM_NODES = 50
NUM_QUBITS = 4
NUM_LAYERS = 3
CLUSTER_SIZE = 10
BYZANTINE_FRACTION = 0.20 # 20% malicious nodes for simulation

def create_heterogeneous_noise_model():
    """Generates a Qiskit noise model based on Table 5 distributions."""
    # Sample from Gaussian distributions as per Table 5
    t1 = np.random.normal(85.0, 15.5) * 1e-6  # Relaxation Time (T1) in seconds
    t2 = np.random.normal(60.0, 12.0) * 1e-6  # Dephasing Time (T2) in seconds
    p_depol = np.random.normal(1.2e-3, 4e-4)  # Local Depolarizing Rate (p)

    # Bound values to prevent unphysical negative times/probabilities
    t1, t2, p_depol = max(t1, 1e-7), max(t2, 1e-7), max(p_depol, 0.0)

    noise_model = NoiseModel()

    # Single-qubit errors
    error_1q = depolarizing_error(p_depol, 1)
    noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz'])

    # Two-qubit errors (CNOT) - standard 98.50% fidelity +/- 0.85%
    p_cnot = 1 - np.random.normal(0.9850, 0.0085)
    p_cnot = max(min(p_cnot, 1.0), 0.0)
    error_2q = depolarizing_error(p_cnot, 2)
    noise_model.add_all_qubit_quantum_error(error_2q, ['cx'])

    return noise_model

# Initialize the 50 edge nodes with unique noisy devices
edge_devices = []
for i in range(NUM_NODES):
    noise = create_heterogeneous_noise_model()
    # PennyLane-Qiskit plugin allows running circuits through Qiskit Aer noise models
    dev = qml.device('qiskit.aer', wires=NUM_QUBITS, noise_model=noise)
    edge_devices.append(dev)

In [6]:
def feature_encoding(x):
    """Non-linear angle encoding mapping data to Hilbert space (Eq. 1)."""
    for i in range(NUM_QUBITS):
        # Use torch.arctan instead of np.arctan to maintain the PyTorch gradient graph
        qml.RY(torch.arctan(x[i]), wires=i)
        qml.RZ(torch.arctan(x[i]**2), wires=i)

def variational_ansatz(weights):
    """L alternating layers of rotations and multi-qubit entangling gates (Eq. 2)."""
    for l in range(NUM_LAYERS):
        for j in range(NUM_QUBITS):
            qml.RY(weights[l, j], wires=j)
            qml.RZ(weights[l, j + NUM_QUBITS], wires=j)
        # U_ent CNOT ladder
        for j in range(NUM_QUBITS - 1):
            qml.CNOT(wires=[j, j+1])
        qml.CNOT(wires=[NUM_QUBITS - 1, 0])

# Define the base circuit for optimization
def qnn_circuit(weights, x):
    feature_encoding(x)
    variational_ansatz(weights)
    return qml.expval(qml.PauliZ(0)) # Measuring expectation value of Pauli-Z on first qubit

def create_mitigated_qnode(device):
    """Wraps the QNode with PennyLane's ZNE transform (Algorithm 3)."""
    base_qnode = qml.QNode(qnn_circuit, device, interface='torch')

    # Apply Richardson extrapolation using odd scale factors (r=1, 3, 5)
    scale_factors = [1, 3, 5]

    # Access ZNE tools directly through the qml top-level namespace
    mitigated_qnode = qml.mitigate_with_zne(
        base_qnode,
        scale_factors=scale_factors,
        folding=qml.fold_global,
        extrapolate=qml.richardson_extrapolate
    )
    return mitigated_qnode

# Initialize mitigated QNodes for all devices
mitigated_nodes = [create_mitigated_qnode(dev) for dev in edge_devices]

In [7]:
def calculate_parameter_fidelity(theta_g, theta_i):
    """
    Approximates quantum trace fidelity based on parameter distance.
    In a physical implementation, this uses the SWAP test (Eq. 8).
    For simulation efficiency, we map the Euclidean parameter distance to a pseudo-fidelity metric.
    """
    distance = torch.norm(theta_g - theta_i)
    # Exponential decay mapping distance to fidelity [0, 1]
    fidelity = torch.exp(-distance)
    return fidelity.item()

def adaptive_threshold_tuning(fidelities, eta_est):
    """Algorithm 4: Adaptive qPBFT Threshold Tuning."""
    if not fidelities:
        return 0.5 # Fallback

    f_avg = np.mean(fidelities)
    f_std = np.std(fidelities) if len(fidelities) > 1 else 0.1

    # Initialize with permissive threshold
    tau = max(0.0, f_avg - 3 * f_std)
    quorum_req = int(np.ceil(2 * eta_est * NUM_NODES)) + 1

    tau_opt = tau
    step_size = 0.01

    # Increment threshold until safety bound is breached
    while tau <= 1.0:
        pool_size = sum(1 for f in fidelities if f >= tau)
        if pool_size < quorum_req:
            break
        tau_opt = tau
        tau += step_size

    return tau_opt

def qpbft_consensus(global_weights, proposed_updates, eta_est):
    """Algorithm 2: Fidelity verification and parameter aggregation."""
    # Phase 3: Fidelity Verification
    fidelities = [calculate_parameter_fidelity(global_weights, update) for update in proposed_updates]

    # Phase 4: Dynamic Threshold & Commit
    tau_opt = adaptive_threshold_tuning(fidelities, eta_est)

    accepted_updates = []
    for i, update in enumerate(proposed_updates):
        if fidelities[i] >= tau_opt:
            accepted_updates.append(update)

    # Aggregation
    if len(accepted_updates) > 0:
        stacked = torch.stack(accepted_updates)
        new_global_weights = torch.mean(stacked, dim=0)
        return new_global_weights, len(accepted_updates), tau_opt
    else:
        return global_weights, 0, tau_opt # Discard epoch if consensus fails

In [ ]:
# Synthetic Data (Concentric overlapping classes)
def generate_synthetic_data(samples=100):
    # Enforce torch.float64 to match PennyLane's internal 64-bit precision
    X = (torch.rand((samples, NUM_QUBITS), dtype=torch.float64) * np.pi)
    y = torch.where(torch.norm(X - (np.pi/2), dim=1) < 1.5, 1.0, -1.0).to(torch.float64)
    return X, y

# Simulation Hyperparameters
EPOCHS = 50
LEARNING_RATE = 0.1
NUM_BYZANTINE = int(NUM_NODES * BYZANTINE_FRACTION)

# Initialize Global Weights explicitly as torch.float64
global_weights = torch.rand((NUM_LAYERS, NUM_QUBITS * 2), dtype=torch.float64, requires_grad=True)

# Generate decentralized data partitions
node_data = [generate_synthetic_data(samples=10) for _ in range(NUM_NODES)]
loss_fn = torch.nn.MSELoss()

print(f"Starting PBFT-QFML Simulation with {NUM_NODES} nodes ({NUM_BYZANTINE} Malicious).")
history_loss = []

for epoch in range(EPOCHS):
    proposed_updates = []

    # Phase 1: Parallel Local Optimization
    for i in range(NUM_NODES):
        X_local, y_local = node_data[i]

        # Determine if node is Byzantine
        is_malicious = i < NUM_BYZANTINE

        if is_malicious:
            # Malicious Strategy: invert gradient and apply multiplier to reach barren plateau
            malicious_weights = global_weights.detach().clone()
            noise = torch.randn_like(malicious_weights, dtype=torch.float64) * 2.0
            proposed_updates.append(malicious_weights + noise)
            continue

        # Honest Node Training Step (Parameter-Shift via Torch Autograd)
        local_weights = global_weights.detach().clone().requires_grad_(True)
        optimizer = torch.optim.SGD([local_weights], lr=LEARNING_RATE)

        optimizer.zero_grad()
        predictions = torch.stack([mitigated_nodes[i](local_weights, x) for x in X_local])
        loss = loss_fn(predictions, y_local)
        loss.backward()
        optimizer.step()

        proposed_updates.append(local_weights.detach())

    # Phase 2 & 3: qPBFT Consensus
    global_weights, consensus_size, threshold = qpbft_consensus(global_weights, proposed_updates, BYZANTINE_FRACTION)

    # Evaluation on global state
    with torch.no_grad():
        test_X, test_y = generate_synthetic_data(samples=50)
        # Evaluate using an ideal (noiseless) device for validation tracking
        ideal_dev = qml.device('default.qubit', wires=NUM_QUBITS)
        ideal_qnode = qml.QNode(qnn_circuit, ideal_dev, interface='torch')
        val_preds = torch.stack([ideal_qnode(global_weights, x) for x in test_X])
        val_loss = loss_fn(val_preds, test_y).item()

    history_loss.append(val_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | Val Loss: {val_loss:.4f} | Accepted Nodes: {consensus_size}/{NUM_NODES} | Tau: {threshold:.3f}")

# Plotting the Convergence
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), history_loss, label='PBFT-QFML + ZNE (Simulated)', color='blue', linewidth=2)
plt.title(f'Global Loss Convergence under {BYZANTINE_FRACTION*100}% Byzantine Attack')
plt.xlabel('Global Training Epochs')
plt.ylabel('Loss (MSE)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

Starting PBFT-QFML Simulation with 50 nodes (10 Malicious).
